In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
# Make sure 'salary_data_cleaned.csv' is uploaded to your Colab files
df = pd.read_csv('salary_data_cleaned.csv')

# Quick check of the data
print("Data Shape:", df.shape)
# print(df.head())


Data Shape: (742, 28)


In [5]:
# 1. Simplify Job Titles to reduce complexity
def title_simplifier(title):
    if 'data scientist' in title.lower(): return 'data scientist'
    elif 'data engineer' in title.lower(): return 'data engineer'
    elif 'analyst' in title.lower(): return 'analyst'
    elif 'machine learning' in title.lower(): return 'mle'
    elif 'manager' in title.lower(): return 'manager'
    elif 'director' in title.lower(): return 'director'
    else: return 'na'

df['job_simp'] = df['Job Title'].apply(title_simplifier)

# 2. Select columns relevant for prediction
# We ignore raw text columns like 'Job Description' for this numeric model
df_model = df[['avg_salary', 'Rating', 'Size', 'Type of ownership', 'Industry', 
               'Sector', 'Revenue', 'hourly', 'employer_provided', 'job_state', 
               'same_state', 'age', 'python_yn', 'R_yn', 'spark', 'aws', 'excel', 'job_simp']]

# 3. Create Dummy Variables (One-Hot Encoding)
df_dum = pd.get_dummies(df_model)

# 4. Train-Test Split
from sklearn.model_selection import train_test_split

X = df_dum.drop('avg_salary', axis=1)
y = df_dum['avg_salary'].values

# Split: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)

Training features shape: (593, 174)
Testing features shape: (149, 174)


In [6]:
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Initialize models
models = {
    "Linear Regression": LinearRegression(),
    "Lasso Regression": Lasso(alpha=0.1), # Alpha handles regularization
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

# Dictionary to store results
results = []

print("Training models...\n")

for name, model in models.items():
    # Fit the model
    model.fit(X_train, y_train)
    
    # Make predictions
    preds = model.predict(X_test)
    
    # Calculate Metrics
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results.append({
        'Model': name,
        'MAE': mae,
        'R2 Score': r2
    })
    
    print(f"{name} trained.")

Training models...

Linear Regression trained.
Lasso Regression trained.
Random Forest trained.
Gradient Boosting trained.


In [7]:
# Convert results to DataFrame for easy viewing
df_results = pd.DataFrame(results)

# Sort by R2 Score (descending) to see the best model on top
df_results = df_results.sort_values(by='R2 Score', ascending=False)

print("\n--- Model Comparison Results ---")
print(df_results)

# Identify the best model
best_model = df_results.iloc[0]
print(f"\nBest Model for this Numeric Data: {best_model['Model']}")
print(f"Accuracy (R2): {best_model['R2 Score']:.4f}")
print(f"Average Error (MAE): ${best_model['MAE']:.2f}k")


--- Model Comparison Results ---
               Model        MAE  R2 Score
2      Random Forest  12.533092  0.771293
3  Gradient Boosting  18.716897  0.637427
1   Lasso Regression  21.489880  0.517853
0  Linear Regression  21.536733  0.471505

Best Model for this Numeric Data: Random Forest
Accuracy (R2): 0.7713
Average Error (MAE): $12.53k
